# NB50 — Optional TabFM third-model feasibility and license preflight

TabFM can strengthen the diversity of real foundation-model comparisons, but a package/API check is **not** a third-model experiment. Official pretrained weights have their own **non-commercial** license, separate from the code's Apache-2.0 terms. Do **not** set the acknowledgment without reading those conditions.

Official implementation: https://github.com/google-research/tabfm.  This notebook does not bypass license terms, does not install conflicting PyTorch builds and does not silently download checkpoints.

In [ ]:
from pathlib import Path
import sys, json, importlib.util, importlib.metadata as md
from datetime import datetime, timezone
I_HAVE_REVIEWED_TABFM_NONCOMMERCIAL_WEIGHT_TERMS = False
ALLOW_TABFM_CHECKPOINT_DOWNLOAD = False
# Optional output directory: change this to the same Drive directory used in v4.
RESULTS_DIR = Path('/content/drive/MyDrive/CoRe_TFM_Resumable_v2/reports') if Path('/content/drive/MyDrive').exists() else Path.cwd()/'tabfm_preflight'
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
info={'timestamp_utc':datetime.now(timezone.utc).isoformat(),
      'source':'https://github.com/google-research/tabfm',
      'code_license':'Apache-2.0',
      'pretrained_weight_license':'tabfm-non-commercial-v1.0',
      'package_installed':importlib.util.find_spec('tabfm') is not None,
      'terms_reviewed':I_HAVE_REVIEWED_TABFM_NONCOMMERCIAL_WEIGHT_TERMS,
      'download_requested':ALLOW_TABFM_CHECKPOINT_DOWNLOAD,
      'status':'NOT_RUN', 'real_model_cache_count':0}
if info['package_installed']:
    info['package_version']=md.version('tabfm')
    try:
        from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch
        info['api_imports_ok']=True
    except Exception as e:
        info['api_imports_ok']=False
        info['preflight_error_type']=type(e).__name__
if not I_HAVE_REVIEWED_TABFM_NONCOMMERCIAL_WEIGHT_TERMS:
    info['status']='BLOCKED_LICENSE_REVIEW_REQUIRED'
elif not info['package_installed']:
    info['status']='BLOCKED_OFFICIAL_PACKAGE_NOT_INSTALLED'
elif not info.get('api_imports_ok',False):
    info['status']='BLOCKED_API_INCOMPATIBLE'
elif not ALLOW_TABFM_CHECKPOINT_DOWNLOAD:
    info['status']='PACKAGE_PREFLIGHT_ONLY__WEIGHTS_NOT_DOWNLOADED'
else:
    try:
        import torch
        if not torch.cuda.is_available():
            info['status']='BLOCKED_NO_CUDA'
        else:
            pretrained=tabfm_v1_0_0_pytorch.load()
            # Successful load is an API feasibility check, NOT a four-view model evaluation.
            info['status']='CHECKPOINT_LOAD_ONLY__NO_BENCHMARK'
            del pretrained
    except Exception as e:
        info['status']='CHECKPOINT_LOAD_FAILED';info['error_type']=type(e).__name__
(RESULTS_DIR/'V4_TABFM_PREFLIGHT.json').write_text(json.dumps(info,indent=2))
print(json.dumps(info,indent=2))


## Requirements to count TabFM as a third TFM

Only count TabFM after it completes **all four predictive views** using the **same frozen dataset–seed–fold IDs**, with model revision and the validated inference arrays archived. Merely importing `TabFMClassifier` or loading pretrained weights must never be labeled a real benchmark. Where classes exceed TabFM's supported maximum, record an explicit model/dataset exclusion; never silently truncate classes.